# **For ICANS Group**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

## **Source EEG based dataframe creation**
**Column names**

**BDSPPatientID, StudyID, Start Time Source File, SourceFilename, ErroneousBDSPInFile, ICANS, ICANSOnsetDate, NoOfDistinctSourceFiles**

In [ ]:
df_icans = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/DailyICANSData.xlsx')

df_icans_filenames = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/ICANSFiles.xlsx')


# ----------------------Select columns that start with 'Unnamed' and Remove them--------------------------------
unnamed_cols = [col for col in df_icans_filenames.columns if col.startswith('Unnamed')]
df_icans_filenames = df_icans_filenames.loc[:, ~df_icans_filenames.columns.str.startswith('Unnamed')]

#------------------- sort by Study IDs in assending order -------------------------
df_icans_filenames = df_icans_filenames.sort_values(by='StudyID', ascending=True).reset_index(drop=True)

#------------------------ Finding ICANS onset dates for each patient ----------------------------------
print('\n')
print('Finding ICANS onset dates for each patient')
df_icans['DateICANSOnset'] = pd.to_datetime(df_icans['DateICANSOnset'], errors='coerce')
# Group by StudyID, drop NaT, keep only date, join as comma-separated string
dates_per_study = (
    df_icans
    .groupby('StudyID')['DateICANSOnset']
    .apply(lambda x: ', '.join(x.dropna().dt.date.astype(str)))
    .reset_index()
)
dates_per_study.columns = ['StudyID', 'ICANS_Onset_Dates']

# Replace empty strings (no dates) with 'Unknown'
dates_per_study['ICANS_Onset_Dates'] = dates_per_study['ICANS_Onset_Dates'].replace('', 'Unknown')

# Count how many are unknown vs known
summary = dates_per_study['ICANS_Onset_Dates'].apply(lambda x: 'Unknown' if x == 'Unknown' else 'Known').value_counts()
total_patients = len(dates_per_study)
print(f"Total patients: {total_patients}\n")
print(summary)


# ----------------------------- Adding the ICANS Onset Date -----------------------------
# Create a dictionary mapping StudyID → ICANS_Onset_Dates
study_to_dates = dict(zip(dates_per_study['StudyID'], dates_per_study['ICANS_Onset_Dates']))

# Add new column to df_icans_filenames using the mapping
df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['StudyID'].map(study_to_dates)

def clean_icans_dates(dates_str):
    # If the value is NaN or Unknown, keep as is
    if pd.isna(dates_str) or dates_str == 'Unknown':
        return dates_str
    
    # Split by comma, remove extra spaces
    dates = [d.strip() for d in dates_str.split(',')]    
    # Get unique dates
    unique_dates = sorted(set(dates))   
    # Join back
    return ', '.join(unique_dates)

df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['ICANSOnsetDate'].apply(clean_icans_dates)

df_icans_filenames = df_icans_filenames.drop(
    columns=["Start Time Segments File", "Segments Filename"],
    errors="ignore"
)

# Finding unique source file per patient
df_icans_filenames["NoOfDistinctSourceFiles"] = (
    df_icans_filenames.groupby("StudyID")["SourceFilename"]
    .transform("nunique")
)

print('\nTotal unique subjects by BDSPID =='+str(df_icans_filenames['BDSPPatientID'].nunique()))
print('Total unique subjects by StidyID =='+str(df_icans_filenames['StudyID'].nunique())+'\n')
df_icans_filenames

## **Adding total No of vanilla valid 10 min segmensts from the SourceEEGFile for every patient**
**Added column names: 'VanillaValidSegmentsFilename', 'NoOfValidVanillaSegments'**

In [ ]:
base_path_to_vanilla_segments = Path(
    "/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/ICANS_group")

def get_segments_for_row(study_id, source_filename):
    # Extract timestamp from source filename
    match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{6})$', source_filename)
    if not match:
        return []
    
    date_part = match.group(1)
    time_part = match.group(2)
    time_formatted = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    timestamp_for_seg = f"{date_part} {time_formatted}"
    
    # Glob all segments for this study in main folder
    all_seg_files = base_path_to_vanilla_segments.glob(f"{study_id}_seg*_SourceStart_*.mat")
    
    # Filter manually by timestamp and only keep the filename
    matched_files = []
    for f in all_seg_files:
        fname_clean = f.name.replace(":", "").replace(" ", "")
        timestamp_clean = timestamp_for_seg.replace(":", "").replace(" ", "")
        if timestamp_clean in fname_clean:
            matched_files.append(f.name)  # <--- only the filename, not full path
    
    return sorted(matched_files)

# Apply to the entire dataframe
df_icans_filenames['VanillaValidSegmentsFilename'] = df_icans_filenames.apply(
    lambda row: get_segments_for_row(row['StudyID'], row['SourceFilename']),
    axis=1
)

df_icans_filenames['NoOfValidVanillaSegments'] = df_icans_filenames['VanillaValidSegmentsFilename'].apply(len)

# Drop rows where NoOfValidVanillaSegments is 0
df_icans_filenames_filtered = df_icans_filenames[df_icans_filenames['NoOfValidVanillaSegments'] != 0].reset_index(drop=True)


print('\nTotal unique subjects by BDSPID =='+str(df_icans_filenames_filtered['BDSPPatientID'].nunique()))
print('Total unique subjects by StidyID =='+str(df_icans_filenames_filtered['StudyID'].nunique())+'\n')
df_icans_filenames_filtered

## **selecting Random Awake Segments**

For the ICANS sensitivity analysis, we randomly selected up to two quality-passing 10-minute EEG segments from each source EEG file, irrespective of ICANS grade (0–4).

- **≤2 valid segments:** All available segments were retained.
- **>2 valid segments:** Two segments were randomly selected from those containing >40% MORGOTH-predicted awake epochs (`pred_class == 0`).
- **Fewer than two eligible awake segments:** The remaining segments were randomly selected from other quality-passing segments to obtain two segments per source file.
- **Reproducibility:** Random selection was performed using a fixed seed of 42.

This strategy evaluates the robustness of NESI–ICANS associations to alternative EEG segment selection while reducing potential bias introduced by selecting segments with maximal awake activity.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

csv_base_path = Path("/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/MORGOTH_SLEEP_OUTPUTS_to_select_top_snippets/ICANS_group")
rng = np.random.default_rng(42)

df_icans_random = df_icans_filenames_filtered.copy()
selected_all = []
awake_cache = {}

for _, row in tqdm(df_icans_random.iterrows(), total=len(df_icans_random)):
    segments = list(row['VanillaValidSegmentsFilename'])

    if len(segments) <= 2:
        selected_all.append(segments)
        continue

    eligible, remaining = [], []

    for seg in segments:
        csv_name = seg.replace('.mat', '.csv')
        csv_path = csv_base_path / csv_name

        if csv_name not in awake_cache:
            if csv_path.exists():
                df_csv = pd.read_csv(csv_path, usecols=['pred_class'])
                awake_cache[csv_name] = (df_csv['pred_class'] == 0).mean()
            else:
                awake_cache[csv_name] = np.nan
                print("NOT FOUND:", csv_name)

        if awake_cache[csv_name] > 0.40:
            eligible.append(seg)
        else:
            remaining.append(seg)

    if len(eligible) >= 2:
        selected = rng.choice(eligible, 2, replace=False).tolist()
    else:
        selected = eligible.copy()
        selected += rng.choice(remaining, 2 - len(selected), replace=False).tolist()

    selected_all.append(selected)

df_icans_random['RandomAwakeSnippetsFilename'] = selected_all

print("Total selected segments:",
      df_icans_random['RandomAwakeSnippetsFilename'].apply(len).sum())

print("Segments per source file:")
print(df_icans_random['RandomAwakeSnippetsFilename'].apply(len).value_counts().sort_index())

# Count number of best segments per row
df_icans_random['NoOfRandomAwakeSegments'] = df_icans_random['RandomAwakeSnippetsFilename'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df_icans_random

# **For Control Group**

## **Source EEG based dataframe creation**
**Column names**

**BDSPPatientID, StudyID, Start Time Source File, SourceFilename, ErroneousBDSPInFile, ICANS, ICANSOnsetDate, NoOfDistinctSourceFiles**

In [ ]:
from pathlib import Path
import pandas as pd

folder_path = Path("/home/ayush/Desktop/ICANS_dataset/AllContinuousEEGs/SourceFullEEGs/NormFullEEGs")

filenames = sorted(f.stem for f in folder_path.iterdir() if f.is_file())

df_controled = pd.DataFrame({
    'SourceFilename': filenames,
    'StudyID': [name.split('_')[0] for name in filenames]
})

df_controled['NoOfDistinctSourceFiles'] = (
    df_controled.groupby('StudyID')['SourceFilename']
    .transform('nunique')
)
df_controled['ICANS'] = 0
print("Total files:", len(df_controled))
print("Unique StudyIDs:", df_controled['StudyID'].nunique())

## **Adding total No of vanilla valid 10 min segmensts from the SourceEEGFile for every patient**


In [ ]:
base_path_to_vanilla_segments = Path(
    "/home/ayush/Desktop/ICANS_dataset/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/Normal_group")

def get_segments_for_row(study_id, source_filename):
    # Extract timestamp from source filename
    match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{6})$', source_filename)
    if not match:
        return []
    
    date_part = match.group(1)
    time_part = match.group(2)
    time_formatted = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    timestamp_for_seg = f"{date_part} {time_formatted}"
    
    # Glob all segments for this study in main folder
    all_seg_files = base_path_to_vanilla_segments.glob(f"{study_id}_seg*_SourceStart_*.mat")
    
    # Filter manually by timestamp and only keep the filename
    matched_files = []
    for f in all_seg_files:
        fname_clean = f.name.replace(":", "").replace(" ", "")
        timestamp_clean = timestamp_for_seg.replace(":", "").replace(" ", "")
        if timestamp_clean in fname_clean:
            matched_files.append(f.name)  # <--- only the filename, not full path
    
    return sorted(matched_files)

# Apply to the entire dataframe
df_controled['VanillaValidSegmentsFilename'] = df_controled.apply(
    lambda row: get_segments_for_row(row['StudyID'], row['SourceFilename']),
    axis=1
)

df_controled['NoOfValidVanillaSegments'] = df_controled['VanillaValidSegmentsFilename'].apply(len)

# Drop rows where NoOfValidVanillaSegments is 0
df_controled_filtered = df_controled[df_controled['NoOfValidVanillaSegments'] != 0].reset_index(drop=True)


print('Total unique subjects by StidyID =='+str(df_controled_filtered['StudyID'].nunique())+'\n')
df_controled_filtered

## **Selecting Random Awake Segments**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

csv_base_path = Path("/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/MORGOTH_SLEEP_OUTPUTS_to_select_top_snippets/Normal_group")
rng = np.random.default_rng(42)

df_control_random = df_controled_filtered.copy()
selected_all = []
awake_cache = {}

for _, row in tqdm(df_control_random.iterrows(), total=len(df_control_random)):
    segments = list(row['VanillaValidSegmentsFilename'])

    if len(segments) <= 2:
        selected_all.append(segments)
        continue

    eligible, remaining = [], []

    for seg in segments:
        csv_name = seg.replace('.mat', '.csv')
        csv_path = csv_base_path / csv_name

        if csv_name not in awake_cache:
            if csv_path.exists():
                df_csv = pd.read_csv(csv_path, usecols=['pred_class'])
                awake_cache[csv_name] = (df_csv['pred_class'] == 0).mean()
            else:
                awake_cache[csv_name] = np.nan
                print("NOT FOUND:", csv_name)

        if awake_cache[csv_name] > 0.40:
            eligible.append(seg)
        else:
            remaining.append(seg)

    if len(eligible) >= 2:
        selected = rng.choice(eligible, 2, replace=False).tolist()
    else:
        selected = eligible.copy()
        selected += rng.choice(remaining, 2 - len(selected), replace=False).tolist()

    selected_all.append(selected)

df_control_random['RandomAwakeSnippetsFilename'] = selected_all

print("Total selected segments:",
      df_control_random['RandomAwakeSnippetsFilename'].apply(len).sum())

print("Segments per source file:")
print(df_control_random['RandomAwakeSnippetsFilename'].apply(len).value_counts().sort_index())

# Count number of best segments per row
df_control_random['NoOfRandomAwakeSegments'] = df_control_random['RandomAwakeSnippetsFilename'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df_control_random

# **Transfer the Best EEG snippets to another folder**

In [ ]:
from tqdm import tqdm
import shutil
from pathlib import Path

# Paths
source_dir = Path("/home/ayush/Desktop/ICANS_dataset/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/Normal_group")
dest_dir = Path("/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/RandomAwakeBasedAnalysis/ICANSRandom10minSegments")
dest_dir.mkdir(parents=True, exist_ok=True)  # create destination folder if it doesn't exist

# Loop over rows with progress bar
for i, row in tqdm(df_control_random.iterrows(), total=len(df_control_random), desc="Copying best segments"):
    best_segments = row['RandomAwakeSnippetsFilename']
    
    if isinstance(best_segments, list) and len(best_segments) > 0:
        for seg_file in best_segments:
            src_file = source_dir / seg_file
            dst_file = dest_dir / seg_file
            
            if src_file.exists():
                shutil.copy(src_file, dst_file)
            else:
                print(f"Source file not found: {src_file}")

# **Create Awake Random Segement Analysis Metadata**

In [ ]:
from pathlib import Path
import pandas as pd
import mat73
import numpy as np

def group_icans(y):
    if y in [0]:
        return 0 # Good Neurological state
    elif y in [1, 2]:
        return 1 # Moderate Neurological state
    elif y in [3, 4]:
        return 2 # Bad Neurological state
    else:
        return None 

folder_path = Path("/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/RandomAwakeBasedAnalysis/ICANSRandom10minSegments")

records = []
for f in sorted(folder_path.glob("*.mat")):
    mat = mat73.loadmat(f, only_include=['ICANS_value'])
    records.append({
        'MorgothOutputFilename': f.with_suffix('.csv').name,
        'BDSPPatientID': f.stem.split('_')[0],
        'RawScore': int(np.asarray(mat['ICANS_value']).squeeze()),
        'Dataset': 'ICANS'
    })

df_icans_morgoth = pd.DataFrame(records)
df_icans_morgoth = df_icans_morgoth[['BDSPPatientID', 'Dataset', 	
                                     'MorgothOutputFilename', 'RawScore']]

df_icans_morgoth["GroupedScore"] = df_icans_morgoth["RawScore"].apply(group_icans)

print("Total files:", len(df_icans_morgoth))
print("Unique patients:", df_icans_morgoth['BDSPPatientID'].nunique())


df_icans_morgoth

In [ ]:
import pandas as pd

csv_path = "/home/ayush/Desktop/NESI_review_material/Morgoth_vs_NESI_datasplit_Realted/Final_NESIscores_full_dataset_result.csv"

df_split = pd.read_csv(csv_path, low_memory=False)
df_split = df_split[df_split['Dataset'] == 'ICANS'].copy()

df_split['BDSPPatientID'] = df_split['BDSPPatientID'].astype(str).str.strip()
df_icans_morgoth['BDSPPatientID'] = df_icans_morgoth['BDSPPatientID'].astype(str).str.strip()

split_counts = df_split.groupby('BDSPPatientID')['Split'].nunique()
assert (split_counts <= 1).all(), "Some patients belong to multiple splits!"

split_map = df_split.drop_duplicates('BDSPPatientID').set_index('BDSPPatientID')['Split']
df_icans_morgoth['Split'] = df_icans_morgoth['BDSPPatientID'].map(split_map)

print("Total rows:", len(df_icans_morgoth))
print("Unique patients:", df_icans_morgoth['BDSPPatientID'].nunique())
print("\nSplit distribution:")
print(df_icans_morgoth.groupby('Split', dropna=False).agg(
    Rows=('BDSPPatientID', 'size'),
    Patients=('BDSPPatientID', 'nunique')
))

In [ ]:
from itertools import combinations

split_ids = {
    s: set(df_icans_morgoth.loc[df_icans_morgoth['Split'] == s, 'BDSPPatientID'])
    for s in ['Train', 'Val', 'Test']
}

for a, b in combinations(split_ids, 2):
    overlap = split_ids[a] & split_ids[b]
    print(f"{a} vs {b}: {len(overlap)} overlapping patients")
    if overlap:
        print("Patient IDs:", sorted(overlap))

In [ ]:
df_icans_morgoth.to_csv('/home/ayush/Desktop/NESI_review_material/ICANS_Awake_Segment_Ablation/RandomAwakeBasedAnalysis/RandomAwake_ICANS_Metadata.csv',
                       index=False)

In [ ]:
df_icans_filenames_filtered.to_csv('/home/ayush/Desktop/Dummy_ICANS.csv', index=False)